# 03.02 · Short-term memory with checkpointers and threads

**Module 03 · Memory & Context Engineering** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 03.01. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Add** short-term memory to an agent with `checkpointer=InMemorySaver()` and a `thread_id`.
2. **Inspect** saved state with `get_state` and `get_state_history`, and explain what a checkpoint is.
3. **Persist** conversations across kernel restarts with `SqliteSaver` (via `mastery.memory.make_checkpointer`).
4. **Keep** custom state fields in the thread, **edit** state with `update_state`, and **delete** a thread.
5. **Avoid** the Windows file-locking traps of SQLite files.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory) | **Main page.** `checkpointer=` on `create_agent`, custom state, access from tools |
| [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) | Threads, checkpoints, super-steps, `get_state`, `get_state_history`, `update_state`, durability, encryption |
| [Checkpointer integrations](https://docs.langchain.com/oss/python/integrations/checkpointers/index) | SQLite, Postgres and other back ends |
| [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html) | Connections, `check_same_thread`, closing files |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "langgraph-checkpoint", "langgraph-checkpoint-sqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/03-memory-and-context/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# Step 5: Develop mastery live (Lesson 03.01's Jupyter skill).
%load_ext autoreload
%autoreload 2

---
## Part 1 · A checkpointer and a thread id

Two things turn memory on:

1. `create_agent(..., checkpointer=saver)`: **where** state is saved.
2. `invoke(input, {"configurable": {"thread_id": "..."}})`: **which** conversation. `mastery.memory.thread_config("...")` builds that dict.

On each call, the agent loads the thread's latest state, **appends** your new message (the `messages` reducer adds, it doesn't replace), runs, and saves.

In [ ]:
# 🧪 Offline: a scripted tutor that answers from what is in its messages.
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from mastery.agents import format_transcript
from mastery.memory import make_checkpointer, thread_config
from mastery.testing import ScriptedChatModel


def recall(messages) -> str:
    """Answer 'what is my name?' from earlier human messages: the model's only 'memory'."""
    said = [m.content for m in messages if isinstance(m, HumanMessage)]
    names = [s.split("I'm ")[1].split()[0].strip(".,!") for s in said if "I'm " in s]
    return f"Your name is {names[-1]}." if names else "I don't know your name yet."


tutor = create_agent(ScriptedChatModel(responses=[recall], repeat_last=True), tools=[],
                     checkpointer=make_checkpointer("memory"))
ana = thread_config("ana-session-1")
tutor.invoke({"messages": [{"role": "user", "content": "Hi, I'm Ana."}]}, ana)
out = tutor.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, ana)
print(format_transcript(out["messages"]))
other = tutor.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, thread_config("someone-else"))
print("\nnew thread →", other["messages"][-1].content)

**You should now see** a 4-line transcript ending in `Your name is Ana.`, and `new thread → I don't know your name yet.`

In [ ]:
# 💸 Live: the same with a real model (2 short calls).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_tutor = create_agent(get_model("fast", temperature=0, reasoning_effort="low"), tools=[],
                              system_prompt="You are a friendly tutor. Answer in one sentence.",
                              checkpointer=make_checkpointer("memory"))
    cfg = thread_config("live-1")
    live_tutor.invoke({"messages": [{"role": "user", "content": "Hi, I'm Ana and fractions confuse me."}]}, cfg)
    reply = live_tutor.invoke({"messages": [{"role": "user", "content": "What's my name and what confuses me?"}]}, cfg)
    print(reply["messages"][-1].content)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** a sentence naming Ana and fractions.

---
## Part 2 · What is saved? `get_state` and `get_state_history`

A **checkpoint** is a snapshot of the whole state, saved after every **super-step** (one round of nodes: input, `model`, `tools`, ...). `get_state(config)` returns the latest snapshot of a thread as a `StateSnapshot`:

| Field | Meaning |
|---|---|
| `values` | The state: `{"messages": [...], ...your fields}` |
| `next` | Nodes that would run next (`()` = finished) |
| `config` | `thread_id`, `checkpoint_ns`, **`checkpoint_id`** of this snapshot |
| `metadata` | `source` (`input`, `loop`, `update`), `step`, ... |
| `created_at` | ISO timestamp (UTC) |
| `parent_config` | The previous checkpoint |

In [ ]:
# 🧪 Offline: the latest snapshot, then the full history (newest first).
snap = tutor.get_state(ana)
print("messages :", len(snap.values["messages"]))
print("next     :", snap.next)
print("step     :", snap.metadata["step"], "| source:", snap.metadata["source"])
print("id       :", snap.config["configurable"]["checkpoint_id"][:13], "…")
print("created  :", snap.created_at)

print("\nhistory (newest first):")
for s in tutor.get_state_history(ana):
    print(f"  step {s.metadata['step']:>2}  source={s.metadata['source']:<6} "
          f"messages={len(s.values.get('messages', []))}  next={s.next}")

**You should now see** `messages : 4`, `next : ()`, `step : 4 | source: loop`, and 6 checkpoints. Read the history bottom-up: for each call there is an `input` checkpoint (your new message not yet merged), a `loop` checkpoint with `next=('model',)`, and a `loop` checkpoint after the model replied (`next=()`). Steps keep counting across calls in the same thread: −1, 0, 1, then 2, 3, 4.

### What just happened?

- Each `invoke` produced **three** checkpoints, not one. That's what makes LangGraph's time travel and human-in-the-loop (Modules 04 and 08) possible: you can resume or fork from any of them.
- The cost: storage grows with every step. Lesson 03.03 keeps the **message list** short; `checkpointer.prune(...)` and `delete_thread(...)` keep the **history** short.
- `created_at` is UTC (ends in `+00:00`). Bogotá is UTC−5, so subtract 5 hours when you read it.

---
## Part 3 · SQLite: memory that survives a kernel restart

`InMemorySaver` lives in the kernel's RAM. Restart the kernel and it's gone. `SqliteSaver` writes the same checkpoints to a file. `mastery.memory.make_checkpointer("sqlite", path)`:

- opens the file with `check_same_thread=False` (Jupyter and LangGraph may touch the connection from different threads; `SqliteSaver` serialises access with its own lock),
- runs `setup()` (creates the `checkpoints` and `writes` tables),
- defaults to `data/outputs/memory/`, which is **git-ignored** (conversations are personal data).

In [ ]:
# 🧪 Offline: a persistent tutor. The thread is saved in a file under data/outputs/memory/.
from mastery.memory import close, memory_dir

DB = memory_dir(REPO_ROOT) / "03-02-tutor.sqlite"
sqlite_saver = make_checkpointer("sqlite", DB)
durable_tutor = create_agent(ScriptedChatModel(responses=[recall], repeat_last=True), tools=[],
                             checkpointer=sqlite_saver)
persist = thread_config("ana-durable")
before = durable_tutor.get_state(persist).values.get("messages", [])
print("messages already in the file:", len(before))
if not before:
    durable_tutor.invoke({"messages": [{"role": "user", "content": "Hi, I'm Ana."}]}, persist)
print("file:", DB.relative_to(REPO_ROOT).as_posix(), f"({DB.stat().st_size:,} bytes)")

**You should now see** `messages already in the file: 0` the first time (and `2` or more on later runs), and the file path `data/outputs/memory/03-02-tutor.sqlite`. The size may look tiny (4,096 bytes): new data first goes to the `-wal` file next to it (Pitfalls table).

**Prove it survives a restart (do this once by hand):**

1. **Kernel → Restart Kernel…**
2. Run Step 0 cells, then the cell above again. It now prints `messages already in the file: 2`.

Restart & Run All can't restart *itself*, so the next cell proves the same thing automatically: a **brand-new Python process** opens the file and reads the thread.

In [ ]:
# 🧪 Offline: a separate Python process (like a restarted kernel) reads the same thread.
import subprocess
import textwrap

reader = textwrap.dedent(f"""
    import sqlite3
    from langgraph.checkpoint.sqlite import SqliteSaver
    saver = SqliteSaver(sqlite3.connect({str(DB)!r}, check_same_thread=False))
    tup = saver.get_tuple({{"configurable": {{"thread_id": "ana-durable"}}}})
    print("other process sees", len(tup.checkpoint["channel_values"]["messages"]), "messages")
    saver.conn.close()
""")
result = subprocess.run([sys.executable, "-c", reader], capture_output=True, text=True, timeout=120)
print(result.stdout.strip() or result.stderr[-800:])

**You should now see** `other process sees 2 messages` (more if you've run the lesson before). The conversation lives in the file, not in the kernel.

`sys.executable` is this kernel's Python, so the child process has the same packages (Lesson 02.01's kernel skill). The `!r` turns the Windows path into a valid Python string literal, backslashes included.

In [ ]:
# 🧪 Offline: peek inside the file with plain sqlite3 (read-only queries).
import sqlite3

with sqlite3.connect(DB) as peek:
    tables = [row[0] for row in peek.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    per_thread = peek.execute("SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id").fetchall()
peek.close()  # `with` commits but does NOT close a sqlite3 connection.
print("tables     :", tables)
print("checkpoints:", per_thread)

**You should now see** `tables : ['checkpoints', 'writes']` and `checkpoints: [('ana-durable', 3)]` (more after repeated runs). State is stored as serialized blobs, not readable rows: always read it through `get_state`.

> **Gotcha:** `with sqlite3.connect(...) as conn:` only manages a **transaction**. It does not close the file. On Windows an open connection keeps the file locked, hence the explicit `close()`.

---
## Part 4 · Custom state, editing state, deleting a thread

Everything in the agent state is checkpointed, not only messages. Add fields with an `AgentState` subclass (Lesson 02.04), and they're saved per thread too.

You can also **edit** a thread from outside a run with `agent.update_state(config, values)`. The update goes through the same reducers: for `messages`, a `RemoveMessage(id=...)` deletes one message. And `checkpointer.delete_thread(thread_id)` removes a whole thread: the "forget this conversation" button.

In [ ]:
# 🧪 Offline: a custom field that persists with the thread; then edit and delete.
from langchain.agents import AgentState
from langchain.messages import AIMessage, RemoveMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


class TutorState(AgentState):
    topics_covered: list[str]


@tool
def mark_topic_done(topic: str, runtime: ToolRuntime) -> Command:
    """Record that a topic was covered in this tutoring session."""
    done = [*runtime.state.get("topics_covered", []), topic]
    return Command(update={"topics_covered": done,
                           "messages": [ToolMessage(f"Covered: {', '.join(done)}", tool_call_id=runtime.tool_call_id)]})


script = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "mark_topic_done", "args": {"topic": "equivalent fractions"}, "id": "t1"}]),
    "Great work on equivalent fractions!",
    AIMessage("", tool_calls=[{"name": "mark_topic_done", "args": {"topic": "adding fractions"}, "id": "t2"}]),
    "Adding fractions: done too.",
])
saver = make_checkpointer("memory")
session_agent = create_agent(script, tools=[mark_topic_done], state_schema=TutorState, checkpointer=saver)
cfg = thread_config("ana-topics")
session_agent.invoke({"messages": [{"role": "user", "content": "We did equivalent fractions."}], "topics_covered": []}, cfg)
session_agent.invoke({"messages": [{"role": "user", "content": "And adding fractions."}]}, cfg)  # no topics_covered: kept
state = session_agent.get_state(cfg).values
print("topics_covered:", state["topics_covered"], "| messages:", len(state["messages"]))

# Edit: delete the first human message (e.g. the user asks to remove something they said).
first = state["messages"][0]
session_agent.update_state(cfg, {"messages": [RemoveMessage(id=first.id)]})
print("after RemoveMessage:", len(session_agent.get_state(cfg).values["messages"]), "messages")

# Forget: delete every checkpoint of the thread.
saver.delete_thread("ana-topics")
print("after delete_thread:", session_agent.get_state(cfg).values)

**You should now see** `topics_covered: ['equivalent fractions', 'adding fractions'] | messages: 8`, then `after RemoveMessage: 7 messages`, then `after delete_thread: {}`.

### What just happened?

- The second `invoke` didn't pass `topics_covered`, yet the list kept the first topic: the field was **loaded from the checkpoint**. Only keys you pass are updated.
- `update_state` wrote a **new checkpoint** (source `update`); the old ones still exist in the history. "Delete" in a checkpointer means "the latest state doesn't have it". To really erase data, delete the thread.
- Removing messages can leave an invalid history (for example a `ToolMessage` whose `AIMessage` is gone). Lesson 03.03 covers safe deletion.

---
## Part 5 · Durability: when are checkpoints written?

`invoke(..., durability=...)` chooses when the checkpointer writes (Checkpointers → Durability modes):

| Mode | Writes | Trade-off |
|---|---|---|
| `"exit"` | Only when the run ends (success, error, interrupt) | Fastest; a crash mid-run loses the run's steps |
| `"async"` | In the background while the next step runs | Good default for most apps |
| `"sync"` | Before the next step starts | Safest; slowest |

For a chat tutor, the default is fine. Long tool-heavy runs (Module 08) are where this choice matters.

In [ ]:
# 🧪 Offline: with durability="exit" a run writes far fewer checkpoints.
for mode in ("sync", "exit"):
    s = make_checkpointer("memory")
    a = create_agent(ScriptedChatModel(responses=[recall], repeat_last=True), tools=[], checkpointer=s)
    a.invoke({"messages": [{"role": "user", "content": "Hi, I'm Ana."}]}, thread_config("d"), durability=mode)
    print(f"{mode:<5}: {len(list(a.get_state_history(thread_config('d'))))} checkpoint(s)")

**You should now see** `sync : 3 checkpoint(s)` and `exit : 1 checkpoint(s)`.

---
## ⚠️ Pitfalls and troubleshooting (Windows first)

| Symptom | Cause | Fix |
|---|---|---|
| `ValueError: Checkpointer requires ... 'thread_id'` | `invoke` without a config | Pass `thread_config("...")` |
| `ProgrammingError: SQLite objects created in a thread can only be used in that same thread` | `sqlite3.connect(path)` without `check_same_thread=False` | Use `make_checkpointer("sqlite", ...)` or pass `check_same_thread=False` |
| `PermissionError: [WinError 32] ... being used by another process` when deleting the `.sqlite` file | A kernel (or DB Browser) still has it open | `close(sqlite_saver)` or restart the kernel, then delete |
| Extra files `*.sqlite-wal` and `*.sqlite-shm` | `SqliteSaver` turns on SQLite's WAL journal | Normal. Delete them together with the `.sqlite` file, only when nothing has it open |
| `database is locked` | Two kernels writing the same file | One file per notebook (or per process); Postgres for multi-process servers |
| Odd errors with a DB in OneDrive / a network drive | Sync tools lock or copy files mid-write | Keep the repo (and `data/outputs/`) outside OneDrive, e.g. `C:\\Users\\you\\code\\` |
| The thread keeps growing forever | Every message is kept | Lesson 03.03 (trim, delete, summarize) |
| State has a field from an old version of your code | Checkpoints outlive code changes | Use a new thread id, or give new fields defaults in your code (`state.get("x", default)`) |

Clean up from **Git Bash** (repo root, kernels closed):
```bash
ls data/outputs/memory/
rm -f data/outputs/memory/03-02-tutor.sqlite*      # the * also removes -wal and -shm
```

## ⚠️ Production notes

> ⚠️ **Production note: SQLite is for one process.**
> A notebook, a CLI or a small single-worker app: SQLite is great. A web server with several workers: use `PostgresSaver` (`langgraph-checkpoint-postgres`, Module 13), or let LangSmith Deployment manage persistence for you.

> ⚠️ **Production note: encrypt and expire conversations.**
> Checkpoints contain everything the user said. `EncryptedSerializer.from_pycryptodome_aes()` (reads `LANGGRAPH_AES_KEY`) encrypts them at rest. Pair it with a retention policy: `prune()` old checkpoints and `delete_thread()` on request.

> ⚠️ **Production note: thread ids belong to users.**
> Store `(user_id, thread_id)` in your own table and check ownership on every request. A guessable thread id (`"ana-1"`) is an access-control bug.

## 🏋️ Exercises

1. **(Easy)** Run Part 1 with a third question, *"I'm Bruno now."*, then *"What's my name?"*. What does `recall` answer, and why?
2. **(Medium)** Write `list_threads(db_path)` that returns the distinct `thread_id`s in a SQLite checkpoint file (read-only SQL, close the connection).
3. **(Medium)** Use `get_state_history` to find the checkpoint just **before** the model's last reply, and call `tutor.invoke(None, that_snapshot.config)`. What happens? (Preview of time travel, Module 08.)
4. **(Stretch)** Add `EncryptedSerializer` to a SQLite saver (`pip install pycryptodome`, set `LANGGRAPH_AES_KEY` to 32 random bytes in `.env`). Check that the blobs in the file are no longer readable msgpack.

<details>
<summary><b>Show solutions</b></summary>

**1.** `Your name is Bruno.`: `recall` takes the **last** "I'm …" in the history.

**2.**
```python
def list_threads(db_path):
    conn = sqlite3.connect(db_path)
    try:
        return [row[0] for row in conn.execute("SELECT DISTINCT thread_id FROM checkpoints")]
    finally:
        conn.close()
```

**3.** The graph **re-runs from that point**: the model is called again and a new branch of checkpoints is written. The old reply stays in the history.
</details>

## ✅ Checkpoint

- [ ] I can add a checkpointer and pick a conversation with `thread_id`.
- [ ] I can read `get_state` and `get_state_history`, and explain why one call makes several checkpoints.
- [ ] I can persist a thread in SQLite and prove it survives a restart.
- [ ] I can keep custom fields per thread, edit with `update_state`, and delete a thread.
- [ ] I know the three durability modes.
- [ ] I can delete SQLite files on Windows without `WinError 32`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open [Checkpointer integrations](https://docs.langchain.com/oss/python/integrations/checkpointers/index). Which back ends are maintained by LangChain and which by the community? For one community back end, check its PyPI page: latest release date and supported `langgraph-checkpoint` versions. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `checkpointer=` + `thread_id` = short-term memory. The checkpointer saves state after every super-step.
- `get_state` / `get_state_history` show what's saved; `update_state` edits; `delete_thread` forgets.
- `make_checkpointer("sqlite", path)` makes conversations survive restarts; keep files in git-ignored `data/outputs/`, close them before deleting.

---
← [03-01-memory-concepts.ipynb](03-01-memory-concepts.ipynb) · [Module 03 README](README.md) · [03-03-managing-long-conversations.ipynb](03-03-managing-long-conversations.ipynb) →